# 00 — Baseline: carry-forward TVT_input

Predict TVT in the evaluation zone as the last observed TVT_input value for that well. No ML — this exists only to (a) anchor a Kaggle leaderboard score, (b) prove the submission pipeline works end to end, (c) give every later model a number to beat.

**Inputs**
- `data/raw/test/{WELLNAME}__horizontal_well.csv` — must contain `MD`, `GR`, `TVT_input` (NaN over the evaluation zone).
- `data/raw/sample_submission.csv` — ordering of (`id = {WELLNAME}_{row_index}`, `tvt`) rows.

**Output**
- `submissions/00_carry_forward_submission.csv` (mirroring sample_submission row order).

In [ ]:
from __future__ import annotations
from pathlib import Path
import pandas as pd
import numpy as np

REPO = Path.cwd().resolve()
while not (REPO / 'pyproject.toml').exists():
    REPO = REPO.parent
RAW = REPO / 'data' / 'raw'
OUT = REPO / 'submissions'
OUT.mkdir(exist_ok=True)
print('Repo:', REPO)
print('Raw data dir resolves to:', RAW.resolve())

In [ ]:
sample = pd.read_csv(RAW / 'sample_submission.csv')
sample[['well', 'row_index']] = sample['id'].str.rsplit('_', n=1, expand=True)
sample['row_index'] = sample['row_index'].astype(int)
wells = sample['well'].unique()
print(f'{len(wells)} wells, {len(sample):,} rows in submission')

In [ ]:
def predict_well(well: str) -> pd.DataFrame:
    h = pd.read_csv(RAW / 'test' / f'{well}__horizontal_well.csv')
    last_known = h['TVT_input'].dropna().iloc[-1] if h['TVT_input'].notna().any() else 0.0
    pred = h['TVT_input'].copy()
    pred[pred.isna()] = last_known
    return pd.DataFrame({
        'id': [f'{well}_{i}' for i in range(len(h))],
        'tvt': pred.values,
    })

preds = pd.concat([predict_well(w) for w in wells], ignore_index=True)
print(preds.head())
print(f'Total predictions: {len(preds):,}')

In [ ]:
submission = sample[['id']].merge(preds, on='id', how='left')
missing = submission['tvt'].isna().sum()
assert missing == 0, f'{missing} ids in sample_submission have no prediction'
out_path = OUT / '00_carry_forward_submission.csv'
submission.to_csv(out_path, index=False)
print('Wrote', out_path, '—', submission.shape)
print(submission['tvt'].describe())

## Next steps

1. Upload `submissions/00_carry_forward_submission.csv` as a Kaggle Notebook submission to confirm the format is accepted.
2. Use the leaderboard score as the floor — every subsequent model must beat it.
3. Move to `notebooks/10_dtw_alignment.ipynb` (DTW-aligned typewell baseline).